# LeetCode 3962 — Maximum Subarray Sum After at Most K Swaps bằng Python

- Link: https://leetcode.com/problems/maximum-subarray-sum-after-at-most-k-swaps/
- Độ khó: **Hard**
- Chủ đề: Mảng, tham lam, sắp xếp, tìm kiếm nhị phân, cấu trúc dữ liệu có thứ tự (`bisect`, Fenwick tree)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSum(self, nums: list[int], k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums` và số `k`. Ta được phép hoán đổi **hai phần tử bất kỳ** (không cần kề nhau) **tối đa `k` lần**. Sau đó lấy mảng con (liên tiếp) có tổng **lớn nhất**. Cần tìm tổng lớn nhất có thể đạt được qua mọi cách hoán đổi và mọi mảng con.

Trực giác: nếu ta đã "định sẵn" mảng con cần lấy, thì mỗi lần hoán đổi chỉ có ích khi **đổi một phần tử nhỏ trong mảng con** lấy **một phần tử lớn ngoài mảng con**. Vì các cặp hoán đổi tốt nhất có cấu trúc rất đẹp (nhỏ nhất bên trong ↔ lớn nhất bên ngoài), ta chỉ cần xử lý từng mảng con, và với `n ≤ 1500` có thể duyệt `~10^6` mảng con nếu mỗi mảng con xử lý nhanh.

Sau notebook này bạn sẽ:

1. Chứng minh/giải thích vì sao với mảng con cố định, ta chỉ cần xét việc "thay một số phần tử bên trong bằng phần tử bên ngoài".
2. Biết cách ghép **nhỏ nhất bên trong** với **lớn nhất bên ngoài** và dừng khi cặp không còn có lợi (gain giảm dần ⇒ tìm kiếm nhị phân số cặp).
3. Dùng danh sách có thứ tự (`bisect.insort`) để duy trì hai tập "trong" và "ngoài" khi mở rộng mảng con.
4. Hiểu thêm phiên bản Fenwick tree (BIT) `O(n² log² n)` của lời giải tham khảo.
5. Viết 4 lời giải (BFS mọi hoán đổi, sắp xếp từng mảng con, danh sách có thứ tự gia tăng, Fenwick tree) và đối chiếu bằng test ngẫu nhiên.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` và `k`. Một **phép hoán đổi** chọn hai chỉ số `i`, `j` bất kỳ và đổi chỗ `nums[i]`, `nums[j]`. Được làm **tối đa `k`** phép. Trả về tổng lớn nhất của một mảng con (không rỗng, liên tiếp) có thể đạt được sau các phép hoán đổi.

**Ví dụ 1:** `nums = [1,-1,0,2]`, `k = 1` → `3`. Hoán đổi chỉ số `1` và `3` được `[1,2,0,-1]`; mảng con `[1,2]` có tổng `3`.

**Ví dụ 2:** `nums = [4,3,2,4]`, `k = 2` → `13` (toàn bộ mảng, không cần hoán đổi).

**Ví dụ 3:** `nums = [-1,-2]`, `k = 0` → `-1` (không được hoán đổi; mảng con tốt nhất là `[-1]`).

**Ràng buộc**

- `1 <= nums.length <= 1500`
- `-10^5 <= nums[i] <= 10^5`
- `0 <= k <= nums.length`

Lưu ý: mảng con phải **không rỗng** (nên đáp án có thể âm, như ví dụ 3), và `k` có thể bằng `0`.

In [1]:
examples = [
    (dict(nums=[1, -1, 0, 2], k=1), 3),
    (dict(nums=[4, 3, 2, 4], k=2), 13),
    (dict(nums=[-1, -2], k=0), -1),
]
for args, expected in examples:
    print("nums =", args["nums"], "| k =", args["k"], "| mong đợi:", expected)

nums = [1, -1, 0, 2] | k = 1 | mong đợi: 3
nums = [4, 3, 2, 4] | k = 2 | mong đợi: 13
nums = [-1, -2] | k = 0 | mong đợi: -1


## 3. Kiến thức nền

### 3.1. Mảng con và tổng mảng con lớn nhất (Kadane)

Mảng con `nums[l..r]` gồm các phần tử liên tiếp. Với `k = 0` bài toán là *Maximum Subarray* kinh điển: thuật toán Kadane giữ `cur` = tổng lớn nhất của mảng con **kết thúc tại vị trí hiện tại**.

In [2]:
def kadane(a):
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)         # hoặc bắt đầu mới tại x, hoặc nối tiếp
        best = max(best, cur)
    return best

print(kadane([1, -1, 0, 2]))      # 2  (mảng con [2], hoặc [1,-1,0,2] = 2)
print(kadane([-1, -2]))           # -1

2
-1


### 3.2. Hoán đổi hai phần tử và nó làm gì với một mảng con cố định

Xét một mảng con cố định `S = nums[l..r]`. Một phép hoán đổi `(i, j)` thuộc một trong ba loại:

1. cả `i, j` **trong** `S`: tổng `S` không đổi;
2. cả `i, j` **ngoài** `S`: tổng `S` không đổi;
3. một chỉ số trong, một ngoài: giá trị trong `S` được thay bằng giá trị ngoài, tổng `S` đổi một lượng `ngoài − trong`.

Chỉ loại 3 có ý nghĩa. Với `t` phép loại 3 ta thay tối đa `t` phần tử trong `S` bằng `t` phần tử ngoài `S`.

In [3]:
a = [5, 1, 9, 2]
inside = (1, 2)             # mảng con a[1..2] = [1, 9], tổng 10
def seg_sum(arr, l, r): return sum(arr[l:r + 1])
b = a[:]; b[1], b[2] = b[2], b[1]          # hoán đổi trong - trong
print(seg_sum(a, *inside), seg_sum(b, *inside))      # 10 10
b = a[:]; b[0], b[3] = b[3], b[0]          # hoán đổi ngoài - ngoài
print(seg_sum(b, *inside))                            # 10
b = a[:]; b[1], b[0] = b[0], b[1]          # trong 1 <-> ngoài 5
print(seg_sum(b, *inside), "= 10 + (5 - 1) =", 10 + (5 - 1))

10 10
10
14 = 10 + (5 - 1) = 14


### 3.3. Ghép nhỏ nhất bên trong với lớn nhất bên ngoài

Muốn tổng `S` tăng nhiều nhất bằng `t` phép loại 3, ta chọn `t` phần tử **nhỏ nhất** trong `S` để loại và `t` phần tử **lớn nhất** ngoài `S` để thêm vào. Ghép chúng: `in_asc[0] ↔ out_desc[0]`, `in_asc[1] ↔ out_desc[1]`, ... Gain của cặp thứ `i` là `out_desc[i] − in_asc[i]`. Vì `in_asc` tăng và `out_desc` giảm nên gain của các cặp **giảm dần**: nếu cặp thứ `i` đã không còn dương (`≤ 0`) thì các cặp sau càng không dương ⇒ dừng.

In [4]:
inside = sorted([3, -2, 7, 0])               # [-2, 0, 3, 7]
outside = sorted([9, 1, -4, 6, 2], reverse=True)   # [9, 6, 2, 1, -4]
T = min(3, len(inside), len(outside))              # tối đa 3 phép
total_gain = 0
for i in range(T):
    g = outside[i] - inside[i]
    print(f"cặp {i}: trong {inside[i]} <-> ngoài {outside[i]}: gain {g}")
    if g <= 0:
        break
    total_gain += g
print("tổng gain tốt nhất:", total_gain)

cặp 0: trong -2 <-> ngoài 9: gain 11
cặp 1: trong 0 <-> ngoài 6: gain 6
cặp 2: trong 3 <-> ngoài 2: gain -1
tổng gain tốt nhất: 17


### 3.4. Danh sách có thứ tự với `bisect`

`bisect.insort(lst, x)` chèn `x` vào danh sách đã sắp xếp (giữ thứ tự) trong `O(log n)` tìm vị trí + `O(n)` dịch chuyển phần tử (nhanh vì là thao tác bộ nhớ ở tầng C). `bisect_left(lst, x)` trả vị trí đầu tiên có thể chèn `x` (nếu `x` có trong list thì là vị trí của nó) — dùng để **xóa** một giá trị: `del lst[bisect_left(lst, x)]`.

In [5]:
from bisect import bisect_left, insort
lst = [1, 4, 4, 9]
insort(lst, 5);  print(lst)                     # [1, 4, 4, 5, 9]
del lst[bisect_left(lst, 4)]; print(lst)        # xóa MỘT số 4 -> [1, 4, 5, 9]
print(lst[:2], lst[-2:], sum(lst[-2:]))         # 2 nhỏ nhất, 2 lớn nhất, tổng 2 lớn nhất

[1, 4, 4, 5, 9]
[1, 4, 5, 9]
[1, 4] [5, 9] 14


### 3.5. Tìm kiếm nhị phân theo "số cặp"

Gain từng cặp giảm dần nên điều kiện "cặp thứ `t` còn dương" là **đơn điệu** theo `t` (đúng cho `t = 1..t*`, sai sau đó). Ta tìm `t*` lớn nhất bằng nhị phân thay vì duyệt tuyến tính.

In [6]:
def last_true(T, pred):
    """t lớn nhất trong [0, T] thỏa pred(t) (pred(0) coi như đúng); pred đơn điệu: đúng rồi sai."""
    lo, hi = 0, T
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if pred(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

inside = [-2, 0, 3, 7]; outside = [-4, 1, 2, 6, 9]       # outside tăng dần
# cặp thứ t (1-based): trong nhỏ thứ t  với  ngoài lớn thứ t
pred = lambda t: inside[t - 1] < outside[-t]
print(last_true(4, pred))   # 3

2


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `nums = [1,-1,0,2]`, `k = 1`.

Thử vài mảng con:

| Mảng con | Tổng | Trong (tăng) | Ngoài (giảm) | Thay `1` cặp tốt nhất | Tổng mới |
|---|---|---|---|---|---|
| `[1]` | 1 | `[1]` | `[2, 0, -1]` | `2 − 1 = +1` | 2 |
| `[1,-1]` | 0 | `[-1, 1]` | `[2, 0]` | `2 − (−1) = +3` | 3 |
| `[1,-1,0]` | 0 | `[-1, 0, 1]` | `[2]` | `2 − (−1) = +3` | 3 |
| `[2]` | 2 | `[2]` | `[1, 0, −1]` | `1 − 2 < 0`: không đổi | 2 |

Cả hai mảng con `[1,-1]` và `[1,-1,0]` cho `3` — khớp đáp án. Điều rút ra: với mảng con cho trước, ta (1) ghép nhỏ nhất trong với lớn nhất ngoài, (2) chỉ làm khi lời, (3) làm tối đa `min(k, kích thước trong, kích thước ngoài)` lần.

Ta có thể kiểm chứng bằng **vét cạn tuyệt đối**: duyệt mọi mảng thu được sau tối đa `k` phép hoán đổi (BFS) rồi chạy Kadane.

## 5. Cách 1 — Vét cạn: BFS qua mọi mảng sau ≤ k phép hoán đổi

**Ý tưởng.** Bắt đầu từ mảng gốc; mỗi bước thử mọi cặp `(i, j)`; thu thập mọi mảng đạt được trong `≤ k` bước (tập `seen`); đáp án là `max` Kadane trên mọi mảng. Đây là định nghĩa **trực tiếp** của đề, dùng làm "chân lý" kiểm tra các cách còn lại.

In [7]:
def brute_force(nums, k):
    n = len(nums)
    start = tuple(nums)
    seen = {start}
    frontier = {start}
    for _ in range(k):
        nxt = set()
        for arr in frontier:
            for i in range(n):
                for j in range(i + 1, n):
                    b = list(arr)
                    b[i], b[j] = b[j], b[i]
                    b = tuple(b)
                    if b not in seen:
                        seen.add(b)
                        nxt.add(b)
        frontier = nxt
        if not frontier:
            break
    return max(kadane(list(arr)) for arr in seen)

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got == expected

3 3
13 13
-1 -1


**Độ phức tạp.** Số mảng có thể là `n!` (hoán vị) ⇒ chỉ dùng cho `n ≤ 6`. Nhưng nó giúp ta tin vào lập luận "mỗi hoán đổi chỉ thay một phần tử trong bằng một phần tử ngoài".

### 5.1. Cách 1b — Với mỗi mảng con: sắp xếp trong/ngoài (`O(n³ log n)`)

Dùng đúng ý tưởng ở 3.3: với mỗi cặp `(l, r)`, sắp xếp phần trong tăng dần, phần ngoài giảm dần, ghép cặp và cộng gain dương (tối đa `T = min(k, c, n − c)` cặp, `c = r − l + 1`).

In [8]:
def per_segment_sort(nums, k):
    n = len(nums)
    best = None
    for l in range(n):
        for r in range(l, n):
            inside = sorted(nums[l:r + 1])
            outside = sorted(nums[:l] + nums[r + 1:], reverse=True)
            total = sum(inside)
            T = min(k, len(inside), len(outside))
            for i in range(T):
                if outside[i] > inside[i]:
                    total += outside[i] - inside[i]
                else:
                    break
            if best is None or total > best:
                best = total
    return best

for args, expected in examples:
    got = per_segment_sort(**args)
    print(got, expected)
    assert got == expected

3 3
13 13
-1 -1


**Độ phức tạp.** `O(n²)` mảng con × `O(n log n)` sắp xếp ⇒ `O(n³ log n)`; với `n = 1500` là `~10^10` — quá chậm. **Nút thắt:** mỗi mảng con ta sắp xếp lại từ đầu dù chỉ khác mảng con trước đúng **một phần tử**.

## 6. Bước đột phá

**Insight 1 — chỉ cần xét "thay trong bằng ngoài".** Như phần 3.2: với mảng con cố định, chỉ hoán đổi trong–ngoài làm đổi tổng; `t` hoán đổi như vậy cho phép thay tối đa `t` phần tử trong bằng `t` phần tử ngoài. (Một dãy hoán đổi dài không thể thay nhiều hơn số phép.)

**Insight 2 — chọn tối ưu.** Với `t` phép, tổng tăng nhiều nhất khi bỏ `t` phần tử nhỏ nhất trong và thêm `t` phần tử lớn nhất ngoài. Gain từng cặp `out_desc[i] − in_asc[i]` giảm dần (hiệu của dãy giảm và dãy tăng), nên dùng nhiều cặp chỉ có lợi **chừng nào gain còn dương**.

**Insight 3 — công thức cho một mảng con.** Với `T = min(k, c, n − c)` và `t*` = số cặp dương đầu tiên (`≤ T`):

`value(l, r) = sum(l..r) + Σ_{i<t*} (out_desc[i] − in_asc[i])`

**Insight 4 — duy trì hai tập khi mở rộng `r`.** Cố định `l`, tăng `r` từng bước: phần tử `nums[r]` chuyển từ **ngoài** sang **trong**. Nếu giữ `inside` (tăng) và `outside` (tăng) trong hai list có thứ tự thì mỗi bước chỉ cần `insort` + xóa một phần tử.

**Insight 5 — tìm `t*` bằng nhị phân.** Điều kiện "cặp thứ `t` còn dương" là `inside[t−1] < outside[−t]` (với `outside` tăng dần, phần tử lớn thứ `t` là `outside[-t]`) — đơn điệu theo `t`, nhị phân `O(log n)`. Tổng gain lấy bằng `sum(outside[-t*:]) − sum(inside[:t*])`.

**Quy tắc "nếu… thì…":**

- Nếu `T = 0` (`k = 0` hoặc mảng con là toàn bộ mảng) thì `value = sum(l..r)`.
- Nếu `inside[0] ≥ outside[-1]` thì `t* = 0`: không có hoán đổi nào có lợi.
- Đáp án = `max` của `value(l, r)` trên mọi `(l, r)`.

## 7. Các cách tiếp theo

### 7.1. Cách 3 — Hai danh sách có thứ tự + nhị phân số cặp (bản chính)

**Giải thích từng đoạn.**

1. Sắp xếp `srt = sorted(nums)` một lần.
2. Với mỗi `l`: `inside = []`, `outside = srt[:]` (bản sao, tất cả phần tử), `s = 0`.
3. Với mỗi `r` từ `l` tới `n − 1`: `x = nums[r]`; `insort(inside, x)`; xóa một `x` khỏi `outside` bằng `del outside[bisect_left(outside, x)]`; `s += x`; `c = r − l + 1`.
4. `T = min(k, c, n − c)`; nhị phân `t*` lớn nhất trong `[0, T]` thỏa `inside[t−1] < outside[−t]`.
5. `value = s + sum(outside[-t*:]) − sum(inside[:t*])` (nếu `t* > 0`); cập nhật đáp án.

In [9]:
def incremental_sorted(nums, k):
    n = len(nums)
    srt = sorted(nums)
    best = None
    for l in range(n):
        inside = []
        outside = srt[:]
        s = 0
        for r in range(l, n):
            x = nums[r]
            insort(inside, x)
            del outside[bisect_left(outside, x)]
            s += x
            c = r - l + 1
            T = min(k, c, n - c)
            lo, hi = 0, T
            while lo < hi:
                mid = (lo + hi + 1) // 2
                if inside[mid - 1] < outside[-mid]:
                    lo = mid
                else:
                    hi = mid - 1
            value = s
            if lo:
                value += sum(outside[-lo:]) - sum(inside[:lo])
            if best is None or value > best:
                best = value
    return best

for args, expected in examples:
    got = incremental_sorted(**args)
    print(got, expected)
    assert got == expected

3 3
13 13
-1 -1


**Độ phức tạp.** `O(n²)` mảng con; mỗi mảng con: `insort`/`del` tốn `O(n)` dịch chuyển bộ nhớ (ở tầng C, rất nhanh), nhị phân `O(log n)`, cộng lát cắt `O(t*)` cũng ở tầng C. Về lý thuyết `O(n³)` thao tác bộ nhớ nhỏ; trong thực tế với `n = 1500` chạy khoảng vài giây trong CPython (ta sẽ đo ở phần 11). Bộ nhớ `O(n)`.

### 7.2. Cách 4 — Fenwick tree (BIT) hai cây: `O(n² log² n)` (lời giải tham khảo)

**Ý tưởng.** Thay hai danh sách có thứ tự bằng **hai cây Fenwick trên giá trị đã nén tọa độ**: mỗi cây lưu đếm và tổng theo giá trị. Từ cây ta lấy được:

- phần tử nhỏ thứ `j` (`kth`) trong `O(log n)`;
- tổng `j` phần tử nhỏ nhất (`sum_smallest`) trong `O(log n)`.

Thêm/xóa một phần tử là `O(log n)` (không dịch chuyển bộ nhớ). Phần tử lớn thứ `t` của tập ngoài là phần tử nhỏ thứ `(d − t + 1)` với `d` là kích thước tập ngoài; tổng `t` phần tử lớn nhất ngoài = tổng toàn bộ − tổng `d − t` phần tử nhỏ nhất.

Mỗi mảng con tốn nhị phân `O(log n)` lần × `kth` `O(log n)` ⇒ `O(log² n)`, tổng `O(n² log² n)`. Trong C++ đây là lựa chọn chuẩn; trong Python hằng số lớn nên cách 3 thường nhanh hơn trong thực tế.

In [10]:
class BIT:
    """Fenwick tree lưu đếm và tổng theo chỉ số giá trị đã nén (1-based)."""
    def __init__(self, m):
        self.m = m
        self.cnt = [0] * (m + 1)
        self.sm = [0] * (m + 1)
        self.top = m.bit_length()
        self.total_cnt = 0
        self.total_sum = 0
    def add(self, i, v, dc):
        self.total_cnt += dc
        self.total_sum += dc * v
        while i <= self.m:
            self.cnt[i] += dc
            self.sm[i] += dc * v
            i += i & -i
    def kth(self, k):
        """chỉ số nhỏ nhất có tổng đếm tiền tố >= k, cùng (đếm, tổng) của các chỉ số đứng trước nó."""
        pos = c = s = 0
        for p in range(self.top, -1, -1):
            np_ = pos + (1 << p)
            if np_ <= self.m and c + self.cnt[np_] < k:
                pos = np_
                c += self.cnt[np_]
                s += self.sm[np_]
        return pos + 1, c, s
    def sum_smallest(self, k, vals):
        if k <= 0:
            return 0
        idx, c, s = self.kth(k)
        return s + (k - c) * vals[idx - 1]

def with_fenwick(nums, k):
    n = len(nums)
    vals = sorted(set(nums))
    pos = {v: i + 1 for i, v in enumerate(vals)}
    best = None
    for l in range(n):
        ins, outs = BIT(len(vals)), BIT(len(vals))
        for x in nums:
            outs.add(pos[x], x, 1)
        s = 0
        for r in range(l, n):
            x = nums[r]
            outs.add(pos[x], x, -1)
            ins.add(pos[x], x, 1)
            s += x
            c, d = r - l + 1, n - (r - l + 1)
            T = min(k, c, d)
            lo, hi = 0, T
            while lo < hi:
                mid = (lo + hi + 1) // 2
                a = vals[ins.kth(mid)[0] - 1]                 # nhỏ thứ mid bên trong
                b = vals[outs.kth(d - mid + 1)[0] - 1]        # lớn thứ mid bên ngoài
                if a < b:
                    lo = mid
                else:
                    hi = mid - 1
            value = s
            if lo:
                value += (outs.total_sum - outs.sum_smallest(d - lo, vals)) - ins.sum_smallest(lo, vals)
            if best is None or value > best:
                best = value
    return best

for args, expected in examples:
    got = with_fenwick(**args)
    print(got, expected)
    assert got == expected

3 3
13 13
-1 -1


**Độ phức tạp.** `O(n² log² n)` thời gian, `O(n)` bộ nhớ. Giá trị `kth` trả về chỉ số nén; ta đổi sang giá trị thật bằng `vals[idx − 1]`.

## 8. TRACE từng bước

Ta trace `incremental_sorted` cho ví dụ 1 (`nums = [1,-1,0,2]`, `k = 1`), chỉ in với `l = 0` và `l = 1`. Mỗi `(l, r)` in: `inside`, `outside` (đều tăng dần), `T`, `t*`, tổng và giá trị.

Bảng mong đợi (một vài dòng):

| `(l, r)` | trong | ngoài | `T` | `t*` | `value` |
|---|---|---|---|---|---|
| (0,0) | `[1]` | `[-1, 0, 2]` | 1 | 1 (`1 < 2`) | `1 + 2 − 1 = 2` |
| (0,1) | `[-1, 1]` | `[0, 2]` | 1 | 1 (`−1 < 2`) | `0 + 2 + 1 = 3` |
| (0,2) | `[-1, 0, 1]` | `[2]` | 1 | 1 | `0 + 2 + 1 = 3` |
| (0,3) | `[-1, 0, 1, 2]` | `[]` | 0 | 0 | `2` |

In [11]:
def trace_incremental(nums, k, only_l=None):
    n = len(nums)
    srt = sorted(nums)
    best = None
    for l in range(n):
        inside, outside, s = [], srt[:], 0
        for r in range(l, n):
            x = nums[r]
            insort(inside, x)
            del outside[bisect_left(outside, x)]
            s += x
            c = r - l + 1
            T = min(k, c, n - c)
            lo, hi = 0, T
            while lo < hi:
                mid = (lo + hi + 1) // 2
                if inside[mid - 1] < outside[-mid]:
                    lo = mid
                else:
                    hi = mid - 1
            value = s + (sum(outside[-lo:]) - sum(inside[:lo]) if lo else 0)
            best = value if best is None else max(best, value)
            if only_l is None or l in only_l:
                print(f"(l={l}, r={r}) trong={inside} ngoài={outside} s={s} T={T} t*={lo} -> value={value}")
    print("đáp án:", best)
    return best

trace_incremental([1, -1, 0, 2], 1, only_l={0, 1})

(l=0, r=0) trong=[1] ngoài=[-1, 0, 2] s=1 T=1 t*=1 -> value=2
(l=0, r=1) trong=[-1, 1] ngoài=[0, 2] s=0 T=1 t*=1 -> value=3
(l=0, r=2) trong=[-1, 0, 1] ngoài=[2] s=0 T=1 t*=1 -> value=3
(l=0, r=3) trong=[-1, 0, 1, 2] ngoài=[] s=2 T=0 t*=0 -> value=2
(l=1, r=1) trong=[-1] ngoài=[0, 1, 2] s=-1 T=1 t*=1 -> value=2
(l=1, r=2) trong=[-1, 0] ngoài=[1, 2] s=-1 T=1 t*=1 -> value=2
(l=1, r=3) trong=[-1, 0, 2] ngoài=[1] s=1 T=1 t*=1 -> value=3
đáp án: 3


3

In [12]:
# Trace một ví dụ k lớn hơn: nhiều cặp có lợi rồi dừng khi gain không dương
trace_incremental([5, -3, -2, 4, 1], 3, only_l={1})
print("vét cạn BFS:", brute_force([5, -3, -2, 4, 1], 3))

(l=1, r=1) trong=[-3] ngoài=[-2, 1, 4, 5] s=-3 T=1 t*=1 -> value=5
(l=1, r=2) trong=[-3, -2] ngoài=[1, 4, 5] s=-5 T=2 t*=2 -> value=9
(l=1, r=3) trong=[-3, -2, 4] ngoài=[1, 5] s=-1 T=2 t*=2 -> value=10
(l=1, r=4) trong=[-3, -2, 1, 4] ngoài=[5] s=0 T=1 t*=1 -> value=8
đáp án: 10
vét cạn BFS: 10


Ở ví dụ thứ hai với `l = 1`, hãy chú ý cách `t*` tăng rồi chững lại: khi các cặp (trong nhỏ nhất, ngoài lớn nhất) bắt đầu cho gain `≤ 0` ta dừng, dù `k` còn dư. Kết quả khớp vét cạn BFS.

## 9. Code nộp LeetCode

Dùng cách 3 (hai danh sách có thứ tự). Viết gọn, đúng chữ ký `maxSum(self, nums, k)`.

In [13]:
from bisect import bisect_left, insort

class Solution:
    def maxSum(self, nums: list[int], k: int) -> int:
        n = len(nums)
        srt = sorted(nums)
        ans = -10**18
        for l in range(n):
            inside = []                     # các phần tử trong nums[l..r], tăng dần
            outside = srt[:]                # các phần tử còn lại, tăng dần
            s = 0
            for r in range(l, n):
                x = nums[r]
                insort(inside, x)
                del outside[bisect_left(outside, x)]
                s += x
                c = r - l + 1
                t_max = min(k, c, n - c)
                lo, hi = 0, t_max           # t lớn nhất có inside[t-1] < outside[-t]
                while lo < hi:
                    mid = (lo + hi + 1) // 2
                    if inside[mid - 1] < outside[-mid]:
                        lo = mid
                    else:
                        hi = mid - 1
                value = s
                if lo:
                    value += sum(outside[-lo:]) - sum(inside[:lo])
                if value > ans:
                    ans = value
        return ans


sol = Solution()
for args, expected in examples:
    got = sol.maxSum(**args)
    print(got, expected)
    assert got == expected

3 3
13 13
-1 -1


## 10. Vì sao thuật toán đúng

**Chỉ trong–ngoài mới có ích.** Cố định mảng con `S`. Sau một dãy `t ≤ k` phép hoán đổi bất kỳ, nội dung `S` (đa tập) bằng đa tập ban đầu với một số phần tử bị thay bằng phần tử từ ngoài. Mỗi phép hoán đổi làm đa tập của `S` thay đổi nhiều nhất **một** phần tử (phần tử bị thay bằng một giá trị khác) nên sau `t` phép, nhiều nhất `t` phần tử của `S` bị thay, và các phần tử thêm vào là `≤ t` phần tử **ngoài** (đa tập các giá trị đến từ ngoài, mỗi vị trí ngoài chỉ đóng góp tối đa một lần theo cách đếm số lượng thay thế). Ngược lại, `t` phần tử trong bất kỳ có thể được thay bằng `t` phần tử ngoài bất kỳ bằng `t` phép hoán đổi rời nhau. Vậy tập các tổng đạt được của `S` bằng tập các tổng nhận được khi thay `j ≤ min(k, c, n − c)` phần tử trong bằng `j` phần tử ngoài.

**Chọn tối ưu.** Để tối đa tổng khi thay đúng `j` phần tử: bỏ `j` nhỏ nhất trong, thêm `j` lớn nhất ngoài (hoán vị tham lam: nếu một cặp được ghép không phải "nhỏ nhất–lớn nhất" thì đổi lại không giảm tổng). Giá trị khi thay `j` phần tử là `G(j) = Σ_{i<j}(out_desc[i] − in_asc[i])`.

**Gain giảm dần ⇒ tối ưu tại `t*`.** Hiệu `out_desc[i] − in_asc[i]` giảm dần theo `i`; `G` lồi lõm (concave): tăng khi hiệu `> 0`, đạt cực đại tại số cặp dương `t*` (hoặc `T` nếu hết quota), rồi không nên tăng thêm. Tức `max_{0≤j≤T} G(j) = G(min(T, số cặp dương liên tiếp đầu))`.

**Nhị phân đúng.** Điều kiện `in_asc[t−1] < out_desc[t−1]` đúng với mọi `t ≤ t*` và sai với `t > t*` (vì hiệu giảm dần): đơn điệu ⇒ nhị phân tìm `t*` lớn nhất. `out_desc[t−1] = outside[−t]` khi `outside` sắp tăng dần.

**Cập nhật gia tăng đúng.** Khi `r` tăng thêm một, `nums[r]` rời tập ngoài và vào tập trong; `insort` giữ `inside` tăng, `del outside[bisect_left(...)]` xóa đúng một bản sao (mọi bản sao bằng nhau nên xóa bản nào cũng như nhau).

**Tối đa trên mọi mảng con.** Đáp án `max_{l ≤ r} value(l, r)`; mảng con không rỗng nên cả `k = 0` và tất cả âm cho đáp án đúng (ví dụ 3).

## 11. Phân tích Time/Space

Đặt `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| BFS mọi hoán đổi | rất lớn (`n!`) | rất lớn |
| Sắp xếp từng mảng con | `O(n³ log n)` | `O(n)` |
| Danh sách có thứ tự (nộp) | `O(n²)` mảng con × (`O(n)` dịch chuyển bộ nhớ + `O(log n)` + cộng lát cắt) ≈ `O(n³)` thao tác bộ nhớ nhỏ, thực tế vài giây | `O(n)` |
| Fenwick tree | `O(n² log² n)` | `O(n)` |

Trong CPython, với `n = 1500` cách 3 chạy trong khoảng vài giây (xem phép đo dưới). Cách Fenwick có độ phức tạp tốt hơn lý thuyết nhưng hằng số lớn hơn nhiều.

In [14]:
import time, random
rng = random.Random(123)
for n in [300, 600, 1500]:
    nums = [rng.randint(-10**5, 10**5) for _ in range(n)]
    t0 = time.time()
    res = Solution().maxSum(nums, n)
    print(f"n={n:>4}, k=n: {res}  ({time.time()-t0:.2f}s)")
nums = [rng.randint(-10**5, 10**5) for _ in range(120)]
t0 = time.time(); a = Solution().maxSum(nums, 30); t1 = time.time(); b = with_fenwick(nums, 30); t2 = time.time()
print(f"n=120,k=30: danh sách {t1-t0:.2f}s, Fenwick {t2-t1:.2f}s, khớp:", a == b)
assert a == b

n= 300, k=n: 7183785  (0.09s)


n= 600, k=n: 14018354  (0.63s)


n=1500, k=n: 37604179  (6.23s)
n=120,k=30: danh sách 0.01s, Fenwick 0.08s, khớp: True


## 12. Bộ test

Chiến lược test:

1. 3 ví dụ của đề (assert).
2. Biên: `k = 0` (Kadane); `n = 1`; mọi phần tử âm; mọi phần tử bằng nhau; `k` rất lớn; mảng đã tối ưu (không cần hoán đổi).
3. Phản ví dụ: hoán đổi "trong–trong" vô ích, ghép sai (lớn nhất–lớn nhất), quên dừng khi gain `≤ 0` (phần 13).
4. Đối chiếu BFS chân lý (`n ≤ 6`), sắp xếp từng mảng con, danh sách có thứ tự, Fenwick và `Solution` trên input ngẫu nhiên (seed cố định).

In [15]:
sol = Solution()
for args, expected in examples:
    assert sol.maxSum(**args) == expected

assert sol.maxSum([5], 3) == 5
assert sol.maxSum([-5, -2, -9], 0) == -2                 # k = 0: Kadane
assert sol.maxSum([-5, -2, -9], 1) == -2                 # mảng con [−2], ngoài có -5, -9: không có lợi
assert sol.maxSum([3, 3, 3], 2) == 9                     # toàn bộ mảng
assert sol.maxSum([1, 2, 3, 4], 0) == 10
assert sol.maxSum([-10, 8, -10, 9], 1) == 17             # mảng con [-10, 8]: thay -10 bằng 9 ngoài mảng con -> 9 + 8
assert sol.maxSum([-10, 8, -10, 9], 2) == 17
assert sol.maxSum([-1, -1, 10, -1, -1, 10], 2) == 20     # mảng con [-1, 10]: thay -1 bằng số 10 ở ngoài -> 10 + 10
print("test cố định OK")

test cố định OK


In [16]:
import random
rng = random.Random(13579)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 7)
    nums = [rng.randint(-6, 8) for _ in range(n)]
    k = rng.randint(0, n)
    if n <= 5 or k <= 2:
        expected = brute_force(nums, k)
        assert per_segment_sort(nums, k) == expected, (nums, k)
        assert incremental_sorted(nums, k) == expected, (nums, k)
        assert Solution().maxSum(nums, k) == expected, (nums, k)
        assert with_fenwick(nums, k) == expected, (nums, k)
        cnt += 1
print("đã đối chiếu", cnt, "input ngẫu nhiên nhỏ: tất cả khớp BFS chân lý")

đã đối chiếu 1256 input ngẫu nhiên nhỏ: tất cả khớp BFS chân lý


In [17]:
# Input vừa: so sánh bốn cách khi BFS không chạy nổi
rng = random.Random(97)
for _ in range(120):
    n = rng.randint(1, 40)
    nums = [rng.randint(-50, 50) for _ in range(n)]
    k = rng.randint(0, n)
    e = per_segment_sort(nums, k)
    assert incremental_sorted(nums, k) == e == Solution().maxSum(nums, k) == with_fenwick(nums, k), (nums, k)
print("4 cách khớp trên 120 input vừa (có giá trị lặp)")

4 cách khớp trên 120 input vừa (có giá trị lặp)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Ghép sai: lấy phần tử lớn nhất bên trong để đổi

**Code sai:** thay các phần tử **lớn** nhất trong mảng con bằng các phần tử lớn nhất ngoài.
**Vì sao sai:** muốn tăng tổng ta phải bỏ phần tử **nhỏ nhất** trong; bỏ phần tử lớn nhất thì ngược lại.
**Code đúng:** `inside` tăng dần, ghép `inside[i]` (nhỏ) với `outside[-1-i]` (lớn).

In [18]:
def wrong_pair_largest(nums, k):
    n = len(nums)
    best = None
    for l in range(n):
        for r in range(l, n):
            inside = sorted(nums[l:r + 1], reverse=True)       # SAI: lớn nhất trước
            outside = sorted(nums[:l] + nums[r + 1:], reverse=True)
            total = sum(inside)
            for i in range(min(k, len(inside), len(outside))):
                if outside[i] > inside[i]:
                    total += outside[i] - inside[i]
                else:
                    break
            best = total if best is None else max(best, total)
    return best

print("sai :", wrong_pair_largest([1, -1, 0, 2], 1), " đúng:", Solution().maxSum([1, -1, 0, 2], 1))

sai : 2  đúng: 3


### Lỗi 2 — Quên dừng khi gain không dương (ép dùng đủ `k` hoán đổi)

**Code sai:** luôn cộng `outside[i] − inside[i]` cho `min(k, c, n − c)` cặp.
**Vì sao sai:** đề cho **tối đa** `k` phép; nếu gain `< 0` thì ta không hoán đổi. Ví dụ: `[4, -2, 4, -5]` với `k = 3` phải ra `8` (mảng con `[4, -2, 4]` giữ nguyên), nhưng ép đổi sẽ ra `6`.
**Code đúng:** `if outside[i] > inside[i]: ... else: break`.

In [19]:
def wrong_force_swaps(nums, k):
    n = len(nums)
    best = None
    for l in range(n):
        for r in range(l, n):
            inside = sorted(nums[l:r + 1])
            outside = sorted(nums[:l] + nums[r + 1:], reverse=True)
            total = sum(inside)
            for i in range(min(k, len(inside), len(outside))):
                total += outside[i] - inside[i]                # SAI: cộng cả gain âm
            best = total if best is None else max(best, total)
    return best

print("sai :", wrong_force_swaps([4, -2, 4, -5], 3), " đúng:", Solution().maxSum([4, -2, 4, -5], 3), " vét cạn:", brute_force([4, -2, 4, -5], 3))

sai : 6  đúng: 8  vét cạn: 8


### Lỗi 3 — Quên giới hạn `min(k, c, n − c)`

**Code sai:** chỉ dùng `T = k`.
**Vì sao sai:** không thể thay nhiều phần tử hơn số phần tử trong mảng con hoặc số phần tử ngoài mảng con ⇒ truy cập ngoài chỉ số hoặc đếm thừa.
**Code đúng:** `T = min(k, c, n - c)`.

### Lỗi 4 — Xóa sai phần tử khỏi danh sách ngoài (xóa theo chỉ số thay vì theo giá trị)

**Code sai:** `del outside[r]` (hoặc `outside.pop(r)`).
**Vì sao sai:** `outside` đã sắp xếp theo giá trị, chỉ số `r` không còn tương ứng vị trí gốc.
**Code đúng:** `del outside[bisect_left(outside, x)]`.

### Lỗi 5 — Dùng `kth` của tập ngoài sai chiều

**Code sai:** lấy "phần tử lớn thứ `t`" của tập ngoài là phần tử nhỏ thứ `t`.
**Vì sao sai:** ngược chiều; phải là nhỏ thứ `d − t + 1` (với `d` là kích thước tập ngoài).

## 14. Mẹo & nhận dạng pattern

- **"Hoán đổi hai phần tử bất kỳ, tối đa `k` lần, tối đa hóa một đại lượng trên một đoạn"** → với đoạn cố định chỉ cần xét đổi **trong ↔ ngoài**; ghép nhỏ nhất–lớn nhất.
- **Gain giảm dần** (hiệu của dãy giảm và dãy tăng) ⇒ dừng sớm / nhị phân số cặp.
- **Duyệt mọi đoạn và duy trì thông tin có thứ tự khi mở rộng một đầu** ⇒ `insort` / BIT.
- `n ≤ 1500` gợi ý `O(n²)` đoạn × chi phí nhỏ (`log`) hoặc chi phí bộ nhớ ở tầng C.
- Khi có `bisect.insort` và `del lst[i]`, cân nhắc chi phí `O(n)` dịch chuyển nhưng hằng số cực nhỏ — thường thắng cấu trúc cây trong Python.
- Dùng vét cạn BFS trên mảng cực nhỏ để kiểm chứng **chính lập luận** ("chỉ cần trong–ngoài"), không chỉ kiểm tra code.

## 15. Biến thể

1. **Tổng mảng con nhỏ nhất sau tối đa `k` hoán đổi:** áp dụng bài này cho `−nums` rồi đổi dấu (Bài 5).
2. **Chỉ được hoán đổi các phần tử kề nhau:** bài khác hẳn (chi phí di chuyển theo khoảng cách) — xem các bài về `k` lần hoán đổi kề.
3. **Mảng con có độ dài cố định `w`:** trượt cửa sổ + hai cấu trúc có thứ tự.
4. **Tối đa tổng `m` phần tử bất kỳ (không cần liên tiếp) sau `k` hoán đổi:** chọn `m` lớn nhất, không cần mảng con.
5. **`k` rất lớn (`k ≥ n`):** thay bằng "chọn mảng con và thay mọi phần tử âm bằng các phần tử dương ngoài".

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi hoán đổi | định nghĩa trực tiếp | `O(n!)` | `O(n!)` | chân lý cho `n ≤ 6` |
| 1b. Sắp xếp từng mảng con | ghép nhỏ nhất–lớn nhất | `O(n³ log n)` | `O(n)` | `n ≤ 100` |
| 3. Danh sách có thứ tự | `insort` + nhị phân `t*` | `≈ O(n³)` bộ nhớ nhỏ, vài giây với `n=1500` | `O(n)` | **nộp bài (Python)** |
| 4. Fenwick tree | `kth` + tổng nhỏ nhất `j` | `O(n² log² n)` | `O(n)` | độ phức tạp tốt nhất lý thuyết (C++/Java) |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `lst = [2, 5, 9]`, chạy `insort(lst, 5)` rồi `del lst[bisect_left(lst, 5)]`. `lst` còn lại là gì?

**Bài 2 (tính tay).** `nums = [2, -5, 3, -1]`, `k = 1`. Đáp án là gì? (Gợi ý: thử mảng con `[2, -5]` và `[-5, 3]`.)

**Bài 3 (điền TODO).** Hoàn thiện `best_gain(inside_sorted, outside_sorted_desc, T)`: tổng gain tốt nhất khi dùng tối đa `T` cặp.

**Bài 4 (tự viết lại).** Viết `my_max_sum(nums, k)` bằng cách sắp xếp từng mảng con (cách 1b).

**Bài 5 (biến thể).** Viết `min_subarray_sum_after_swaps(nums, k)`: tổng mảng con **nhỏ nhất** có thể sau tối đa `k` hoán đổi.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def best_gain(inside_sorted, outside_sorted_desc, T):
    # TODO: tổng (outside[i] - inside[i]) cho i = 0.. khi còn dương, tối đa T cặp
    pass

# Bài 4 — TODO
def my_max_sum(nums, k):
    # TODO: với mỗi (l, r): inside tăng, outside giảm, T = min(k, c, n-c), cộng gain dương
    pass

# Bài 5 — TODO
def min_subarray_sum_after_swaps(nums, k):
    # TODO: gợi ý: -maxSum(-nums)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `insort` chèn vào giữ thứ tự.
- Mức 2: sau `insort(lst, 5)` có `[2, 5, 5, 9]`.
- Mức 3: `bisect_left` trỏ tới số `5` đầu tiên; xóa một số `5` → `[2, 5, 9]`.

**Bài 2.**
- Mức 1: thử `[2, -5]`: trong `[-5, 2]`, ngoài `[3, -1]`.
- Mức 2: cặp `(-5, 3)` gain `+8`; tổng `-3 + 8 = 5`.
- Mức 3: không mảng con nào cho hơn `5` ⇒ `5`.

**Bài 3.**
- Mức 1: `for i in range(T)`.
- Mức 2: nếu `outside[i] > inside[i]` thì cộng, ngược lại `break`.
- Mức 3: trả về tổng.

**Bài 4.**
- Mức 1: hai vòng `l`, `r`.
- Mức 2: `inside = sorted(nums[l:r+1])`, `outside = sorted(nums[:l] + nums[r+1:], reverse=True)`.
- Mức 3: `total = sum(inside) + best_gain(inside, outside, min(k, c, n-c))`.

**Bài 5.**
- Mức 1: tổng nhỏ nhất của mảng con ứng với tổng lớn nhất của mảng đổi dấu.
- Mức 2: hoán đổi không phụ thuộc dấu giá trị.
- Mức 3: `-Solution().maxSum([-x for x in nums], k)`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
lst = [2, 5, 9]
insort(lst, 5)
del lst[bisect_left(lst, 5)]
assert lst == [2, 5, 9]
# Bài 2
assert Solution().maxSum([2, -5, 3, -1], 1) == 5 == brute_force([2, -5, 3, -1], 1)

# Bài 3
def best_gain_ans(inside_sorted, outside_sorted_desc, T):
    total = 0
    for i in range(T):
        if outside_sorted_desc[i] > inside_sorted[i]:
            total += outside_sorted_desc[i] - inside_sorted[i]
        else:
            break
    return total

# cặp (-2, 9): +11; cặp (0, 6): +6; cặp (3, 2): -1 -> dừng  => tổng 17
assert best_gain_ans([-2, 0, 3, 7], [9, 6, 2, 1, -4], 3) == 17
assert best_gain_ans([1, 2], [0, -1], 2) == 0
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_max_sum_ans(nums, k):
    n = len(nums)
    best = None
    for l in range(n):
        for r in range(l, n):
            inside = sorted(nums[l:r + 1])
            outside = sorted(nums[:l] + nums[r + 1:], reverse=True)
            c = r - l + 1
            total = sum(inside) + best_gain_ans(inside, outside, min(k, c, n - c))
            if best is None or total > best:
                best = total
    return best

# Bài 5
def min_subarray_sum_after_swaps_ans(nums, k):
    return -Solution().maxSum([-x for x in nums], k)

def brute_min(nums, k):
    n = len(nums)
    start = tuple(nums); seen = {start}; frontier = {start}
    for _ in range(k):
        nxt = set()
        for arr in frontier:
            for i in range(n):
                for j in range(i + 1, n):
                    b = list(arr); b[i], b[j] = b[j], b[i]; b = tuple(b)
                    if b not in seen:
                        seen.add(b); nxt.add(b)
        frontier = nxt
    def min_sub(a):
        best = cur = a[0]
        for x in a[1:]:
            cur = min(x, cur + x); best = min(best, cur)
        return best
    return min(min_sub(list(a)) for a in seen)

rng = random.Random(31)
for _ in range(300):
    n = rng.randint(1, 5)
    nums = [rng.randint(-7, 7) for _ in range(n)]
    k = rng.randint(0, 3)
    assert my_max_sum_ans(nums, k) == brute_force(nums, k)
    assert min_subarray_sum_after_swaps_ans(nums, k) == brute_min(nums, k), (nums, k)
print("Bài 3–5 OK")

Bài 3–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đoạn cố định + hoán đổi trong–ngoài tối đa `k` lần":**

```python
srt = sorted(nums)
for l in range(n):
    inside, outside, s = [], srt[:], 0
    for r in range(l, n):
        x = nums[r]
        insort(inside, x)
        del outside[bisect_left(outside, x)]
        s += x
        c = r - l + 1
        lo, hi = 0, min(k, c, n - c)
        while lo < hi:                              # số cặp dương lớn nhất
            mid = (lo + hi + 1) // 2
            if inside[mid - 1] < outside[-mid]: lo = mid
            else: hi = mid - 1
        value = s + (sum(outside[-lo:]) - sum(inside[:lo]) if lo else 0)
        ans = max(ans, value)
```

**Checklist trước khi submit:**

- [ ] Với đoạn cố định chỉ xét hoán đổi **trong ↔ ngoài**.
- [ ] Ghép **nhỏ nhất bên trong** với **lớn nhất bên ngoài**.
- [ ] Số cặp tối đa `min(k, c, n − c)`; dừng khi gain `≤ 0`.
- [ ] Xóa khỏi `outside` bằng `bisect_left` theo **giá trị**.
- [ ] `ans` khởi tạo rất nhỏ (đáp án có thể âm, mảng con không rỗng).
- [ ] `k = 0` trả đúng Kadane.
- [ ] Đã đối chiếu BFS chân lý trên mảng `n ≤ 6`.